In [1]:
from crewai import LLM, Crew, Agent, Task
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import SentenceTransformerEmbeddings  # Use the wrapper
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain.agents import tool
from crewai.tools import tool

In [2]:
llm = LLM(model="ollama/llama3.2:1b-instruct-q8_0",base_url="http://localhost:11434")

#llm = LLM(model="groq/openai/gpt-oss-20b")

In [3]:
# Website Data Ingestion 
loader = UnstructuredURLLoader(urls=["https://docs.crewai.com/how-to/Installing-CrewAI/"])
data = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(data)
texts

[Document(metadata={'source': 'https://docs.crewai.com/how-to/Installing-CrewAI/'}, page_content='Skip to main content\n\nCrewAI home page\n\nlight logo\n\ndark logo\n\nHomeDocumentationAMPAPI ReferenceExamplesChangelog\n\nWebsite\n\nForum\n\nBlog\n\nCrewGPT\n\nWelcome\n\nCrewAI Documentation\n\nWelcome\n\nCrewAI Documentation\n\nBuild collaborative AI agents, crews, and flows — production ready from day one.\n\nShip multi‑agent systems with confidence\n\nDesign agents, orchestrate crews, and automate flows with guardrails, memory, knowledge, and observability baked in.\n\nGet startedView changelogAPI Reference\n\nGet started\n\nIntroduction\n\nOverview of CrewAI concepts, architecture, and what you can build with agents, crews, and flows.\n\nInstallation\n\nInstall via uv, configure API keys, and set up the CLI for local development.\n\nQuickstart\n\nSpin up your first crew in minutes. Learn the core runtime, project layout, and dev loop.\n\nBuild the basics\n\nAgents\n\nCompose agent

In [8]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    dimensions=512,
)

_dlist = [d.page_content for d in texts]

_list = embeddings.embed_documents(_dlist)
_list[0]

[0.009238904,
 0.036994666,
 -0.21336749,
 0.023482116,
 0.04644561,
 -0.080838926,
 0.014584793,
 -0.021960722,
 0.002244522,
 -0.09520251,
 -0.0048074336,
 0.045725085,
 0.08571402,
 0.071943246,
 0.015636455,
 0.01192349,
 -0.026223011,
 -0.003921475,
 0.018769419,
 0.040551886,
 -0.021539776,
 -0.0331073,
 0.013912908,
 -0.07642725,
 0.097309634,
 0.026101599,
 -0.013639629,
 -0.044321727,
 -0.051740114,
 -0.037976637,
 0.00048110797,
 0.023695707,
 -0.034660626,
 -0.10585378,
 0.02869257,
 -0.08338119,
 0.040301673,
 0.017535308,
 0.022547929,
 0.04445385,
 -0.0059440336,
 0.014294563,
 -0.01726934,
 -0.08303397,
 -0.0068782456,
 -0.038998943,
 0.004641734,
 -0.054842312,
 0.0558582,
 0.0014413564,
 0.029677343,
 0.010564465,
 -0.024069062,
 -0.034504775,
 0.058270425,
 0.010895664,
 -0.057382215,
 -0.024634544,
 -0.015326398,
 -0.06278566,
 0.08747733,
 0.10236622,
 0.041402377,
 0.121575885,
 0.046295416,
 -0.04482193,
 -0.0377978,
 0.0150503,
 0.029403565,
 0.012178787,
 0.0076

#### This example walks through building a retrieval augmented generation (RAG) application using Ollama and embedding models.

In [ ]:
# Step 1: Generate embeddings
import os
import chromadb

## Create a simple txt file
work_dir = os.getenv("WORK_DIR")
persist_directory = f"{work_dir}/vactor_store/chroma"
client = chromadb.PersistentClient(path=persist_directory)

#
collection = client.get_or_create_collection(name="docs")

In [9]:
# Step 1: Generate embeddings
import ollama

documents = [
  "Llamas are members of the camelid family meaning they're pretty closely related to vicuñas and camels",
  "Llamas were first domesticated and used as pack animals 4,000 to 5,000 years ago in the Peruvian highlands",
  "Llamas can grow as much as 6 feet tall though the average llama between 5 feet 6 inches and 5 feet 9 inches tall",
  "Llamas weigh between 280 and 450 pounds and can carry 25 to 30 percent of their body weight",
  "Llamas are vegetarians and have very efficient digestive systems",
  "Llamas live to be about 20 years old, though some only live for 15 years and others live to be 30 years old",
]

#client = chromadb.Client()
#collection = client.create_collection(name="docs")

# store each document in a vector embedding database
for i, d in enumerate(documents):
  # mxbai-embed-large
  response = ollama.embed(model="nomic-embed-text", input=d)
  embeddings = response["embeddings"]
  collection.add(
    ids=[str(i)],
    embeddings=embeddings,
    documents=[d]
  )

NameError: name 'collection' is not defined

In [ ]:
# Step 2: Retrieve : the most relevant document given an example prompt:

import ollama

prompt = "What animals are llamas related to ?"

# generate an embedding for the input and retrieve the most relevant doc
response = ollama.embed(
  model="all-minilm",
  input=prompt
)

results = collection.query(
  query_embeddings=response["embeddings"],
  n_results=1
)

data = results['documents'][0][0]

In [ ]:
data

In [ ]:
#
# Step 3: Generate : response combining the prompt and data we retrieved in step 2
#
input = "What animals are llamas related to ?"
output = ollama.generate(
  model="llama3.2:1b-instruct-q8_0",
  prompt=f"Using this data: {data}. Respond to this prompt: {input}"
)

print(output['response'])

In [ ]:
from crewai_tools import WebsiteSearchTool
from crewai import Agent, LLM
from crewai.project import agent
from crewai_tools import RagTool
from crewai.tools import tool

@tool
def website_search(http_url: str):
    """
    Useful for when you need to ask with lookup on website data.
    """
    #return retriever.get_relevant_documents()
    # Proper Configuration Structure
    search_tool = WebsiteSearchTool(
        website='https://example.com',
        config={
            "embedder": {
                "provider": "huggingface",
                "config": {
                    "model": "sentence-transformers/all-MiniLM-L6-v2",
                },
            },
        }
    )

    return search_tool.run(http_url)

#website_search("https://docs.crewai.com/how-to/Installing-CrewAI/")   

In [ ]:
import ollama

single = ollama.embed(
  model='all-minilm',
  input='The sky is blue because of Rayleigh scattering'
)
print(single['embeddings'][0])  # vector length
print(len(single['embeddings'][0]))  # vector length

In [4]:
researcher = Agent(
    llm=llm,
    name="Researcher",
    role="Researches topics by searching the website data.",
    tools=[website_search],
    goal="Answer questions by retrieving relevant information from the website's data.",  # Add the goal here
    backstory="You are a helpful AI assistant specializing in searching and retrieving information from a website. Use your 'Website_Search' tool to find relevant documents when answering questions."  
)

researcher_boss = Agent(
    llm=llm,
    name="Researcher Boss",
    role="Challenges the researcher to bring out the best out of his findings",
    tools=[website_search],
    goal="Ask further questions to the researcher and validates the retrieved relevant information from the website's data.",  # Add the goal here
    backstory="You are a helpful AI assistant boss and your job is to make sure the retrieved information is correct. Use your 'Website_Search' tool to find relevant documents when answering questions."  
)


In [5]:
from crewai import Task
research_task = Task(
    description=(
        "Analyze the URL provided ({crewai_url}) "
        "to extract information about how crewai works. "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of crewai specifications, including necessary "
        "tools to get started"
    ),
    agent=researcher,
    #async_execution=True
)

boss_task = Task(
    description=(
        "Analyze the URL provided ({crewai_url}) "
        "to extract information about how crewai works. "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of crewai specifications, including necessary "
        "tools to get started"
    ),
    agent=researcher_boss,
    #async_execution=True
)


In [6]:
from crewai import Crew
# Create Crew
research_crew1 = Crew(
    agents=[researcher, researcher_boss],
    tasks=[research_task, boss_task],
    verbose=True,  # This will print logs to the console as the crew works
    planning=True,
)

# Job Context
job_crew_works = {
    'crewai_url': 'https://docs.crewai.com/how-to/Installing-CrewAI/',
    'personal_writeup': """Accomplished Researcher 
    with 18 years of experience, specializing in
    setting up CrewAI kind of agent based systems"""
}

async_result = await research_crew1.kickoff_async(inputs=job_crew_works)
print(async_result)

# Kickoff the Crew's Work
#research_crew1.kickoff(inputs=job_crew_works)
#print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 56350921-8e18-40f7-af21-87316b552815                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[2026-04-15 11:00:25][INFO]: Planning the crew execution


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - Analyze the URL provided (https://docs.crewai.com/how-to/Installing-CrewAI/)   │
│  to extract information about how crewai works. required. Use the tools to gather content and identify and      │
│  categorize the requirements.                                                                                   │
│                  "task_description": Analyze the URL provided                                                   │
│  (https://docs.crewai.com/how-to/Installing-CrewAI/) to extract information about how crewai works. required.   │
│  Use the tools to gather content and identify and categorize the requirements.                                  │
│                  "task_expected_output": A structured list of crewai specifications, including necessary tools  │
│  to get started                                                                                                 │
│                  "agent": Researches topics by searching the website data.                                      │
│                  "agent_goal": Answer questions by retrieving relevant information from the website's data.     │
│                  "task_tools": [Tool(name='website_search', description='Tool Name: website_search\nTool        │
│  Arguments: {\n  "properties": {\n    "http_url": {\n      "title": "Http Url",\n      "type": "string"\n       │
│  }\n  },\n  "required": [\n    "http_url"\n  ],\n  "title": "Website_Search",\n  "type": "object",\n            │
│  "additionalProperties": false\n}\nTool Description: \n    Useful for when you need to ask with lookup on       │
│  website data.\n    ', env_vars=[], args_schema=<class 'crewai.tools.base_tool.Website_Search'>,                │
│  description_updated=False, cache_function=<function _default_cache_function at 0x7ab1a796cb80>,                │
│  result_as_answer=False, max_usage_count=None, current_usage_count=0, func=<function website_search at          │
│  0x7ab0b8a59bc0>, tool_type='crewai.tools.base_tool.Tool')]                                                     │
│                  "agent_tools": [name='website_search' description='Tool Name: website_search\nTool Arguments:  │
│  {\n  "properties": {\n    "http_url": {\n      "title": "Http Url",\n      "type": "string"\n    }\n  },\n     │
│  "required": [\n    "http_url"\n  ],\n  "title": "Website_Search",\n  "type": "object",\n                       │
│  "additionalProperties": false\n}\nTool Description: \n    Useful for when you need to ask with lookup on       │
│  website data.\n    ' env_vars=[] args_schema=<class 'crewai.tools.base_tool.Website_Search'>                   │
│  description_updated=False cache_function=<function _default_cache_function at 0x7ab1a796cb80>                  │
│  result_as_answer=False max_usage_count=None current_usage_count=0 func=<function website_search at             │
│  0x7ab0b8a59bc0> tool_type='crewai.tools.base_tool.Tool']                                                       │
│                  Task Number 2 - Analyze the URL provided (https://docs.crewai.com/how-to/Installing-CrewAI/)   │
│  to extract information about how crewai works. required. Use the tools to gather content and identify and      │
│  categorize the requirements.                                                                                   │
│                  "task_description": Analyze the URL provided                                                   │
│  (https://docs.crewai.com/how-to/Installing-CrewAI/) to

ERROR:root:OpenAI API call failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
ERROR:root:OpenAI API call failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: OpenAI API call failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota,       │
│  please check your plan and billing details. For more information on this error, read the docs:                 │
│  https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param':       │
│  None, 'code': 'insufficient_quota'}}                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[CrewAIEventsBus] Warning: Ending event 'crew_kickoff_failed' emitted with empty scope stack. Missing starting 
event?

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - Analyze the URL provided (https://docs.crewai.com/how-to/Installing-CrewAI/)   │
│  to extract information about how crewai works. required. Use the tools to gather content and identify and      │
│  categorize the requirements.                                                                                   │
│                  "task_description": Analyze the URL provided                                                   │
│  (https://docs.crewai.com/how-to/Installing-CrewAI/) to extract information about how crewai works. required.   │
│  Use the tools to gather content and identify and categorize the requirements.                                  │
│                  "task_expected_output": A structured list of crewai specifications, including necessary tools  │
│  to get started                                                                                                 │
│                  "agent": Researches topics by searching the website data.                                      │
│                  "agent_goal": Answer questions by retrieving relevant information from the website's data.     │
│                  "task_tools": [Tool(name='website_search', description='Tool Name: website_search\nTool        │
│  Arguments: {\n  "properties": {\n    "http_url": {\n      "title": "Http Url",\n      "type": "string"\n       │
│  }\n  },\n  "required": [\n    "http_url"\n  ],\n  "title": "Website_Search",\n  "type": "object",\n            │
│  "additionalProperties": false\n}\nTool Description: \n    Useful for when you need to ask with lookup on       │
│  website data.\n    ', env_vars=[], args_schema=<class 'crewai.tools.base_tool.Website_Search'>,                │
│  description_updated=False, cache_function=<function _default_cache_function at 0x7ab1a796cb80>,                │
│  result_as_answer=False, max_usage_count=None, current_usage_count=0, func=<function website_search at          │
│  0x7ab0b8a59bc0>, tool_type='crewai.tools.base_tool.Tool')]                                                     │
│                  "agent_tools": [name='website_search' description='Tool Name: website_search\nTool Arguments:  │
│  {\n  "properties": {\n    "http_url": {\n      "title": "Http Url",\n      "type": "string"\n    }\n  },\n     │
│  "required": [\n    "http_url"\n  ],\n  "title": "Website_Search",\n  "type": "object",\n                       │
│  "additionalProperties": false\n}\nTool Description: \n    Useful for when you need to ask with lookup on       │
│  website data.\n    ' env_vars=[] args_schema=<class 'crewai.tools.base_tool.Website_Search'>                   │
│  description_updated=False cache_function=<function _default_cache_function at 0x7ab1a796cb80>                  │
│  result_as_answer=False max_usage_count=None current_usage_count=0 func=<function website_search at             │
│  0x7ab0b8a59bc0> tool_type='crewai.tools.base_tool.Tool']                                                       │
│                  Task Number 2 - Analyze the URL provided (https://docs.crewai.com/how-to/Installing-CrewAI/)   │
│  to extract information about how crewai works. required. Use the tools to gather content and identify and      │
│  categorize the requirements.                                                                                   │
│                  "task_description": Analyze the URL provided                                                   │
│  (https://docs.crewai.com/how-to/Installing-CrewAI/) to

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ c529c5d3-ea5c-43bf-a343-a5fc7b8614fb                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/c529c5d3-ea5c-43b │
│ f-a343-a5fc7b8614fb?access_code=TRACE-ccce5813f8                             │
│ 🔑 Access Code: TRACE-ccce5813f8                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 56350921-8e18-40f7-af21-87316b552815                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
from crewai_tools import RagTool
from crewai_tools.tools.rag import RagToolConfig, VectorDbConfig, ProviderSpec
from crewai.rag.embeddings.providers.openai.types import OpenAIProviderSpec
from crewai.rag.embeddings.providers.ollama.types import OllamaProviderSpec
from crewai.rag.embeddings.providers.huggingface.types import HuggingFaceProviderSpec
# Create a RAG tool with custom configuration

vectordb: VectorDbConfig = {
    "provider": "chromadb",
    "config": {
        "collection_name": "my-collection"
    }
}

embedding_model: OpenAIProviderSpec = {
    "provider": "openai",
    "config": {
        #"api_key": "ollama",
        "model_name": "all-minilm",
        "dimensions": 4096,
        #"organization_id": "your-org-id",
        "api_base": "http://localhost:11434/v1/",
        #"api_version": "v1",
        #"default_headers": {"X-Custom-Header": "xxxxx"}
    }
}

ollama_embedding_model: OllamaProviderSpec = {
    "provider": "ollama",
    "config": {
        "model_name": "all-minilm",
        "url": "http://localhost:11434/api/embeddings"
    }
}

hf_embedding_model: HuggingFaceProviderSpec = {
    "provider": "huggingface",
    "config": {
        "url": "https://api-inference.huggingface.co/models/sentence-transformers/all-MiniLM-L6-v2"
    }
}

config: RagToolConfig = {
    "vectordb": vectordb,
    "embedding_model": embedding_model
}

rag_tool = RagTool(config=config, summarize=True)

In [ ]:
# Add a PDF file
rag_tool.add(data_type="file", path="/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf")

# Add a web page
#rag_tool.add(data_type="web_page", url="https://example.com")

# Add a YouTube video
#rag_tool.add(data_type="youtube_video", url="https://www.youtube.com/watch?v=VIDEO_ID")

# Add a directory of files
#rag_tool.add(data_type="directory", path="/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs")